<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_88_synthetic_data_distillation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🧪 **Module 88 — Synthetic Data Generation + Distillation** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 12 · Production Extensions


# Module 88 — Synthetic Data Generation + Distillation

> The 2024-2026 frontier-model recipe is no longer "scrape more web." It's **synthetic data**: a strong teacher generates training data for a smaller / faster student. **Phi-3 / Phi-4**, **Orca-2**, **WizardLM Evol-Instruct**, **Self-Instruct**, **Self-Rewarding LM**, **Magpie**, **Nemotron-4 340B**, **DeepSeek-R1 → distilled-Qwen / distilled-Llama**, **STaR / Quiet-STaR**, **rejection sampling fine-tuning (RFT)** — all the same idea with different sauces. This module covers the full taxonomy, the math (KL knowledge distillation vs SFT-on-teacher-outputs), the **quality filters** that decide success, and a hands-on Magpie + RFT loop.

### What you'll cover
1. **Why synthetic data** — the data wall, cost of human labels, and the Phi / Orca recipe
2. The four flavors — **SFT distillation · KL distillation · Self-Instruct · RLAIF/RFT**
3. **Self-Instruct** + **Evol-Instruct** (WizardLM) prompt-evolution
4. **Magpie** — the "extract instructions from a chat-LLM's own opening tokens" trick
5. **Orca / Orca-2** — teacher's reasoning trace as the supervision signal
6. **Self-Rewarding LM** + **RLAIF / Constitutional AI** as data engines
7. **Rejection Sampling Fine-Tuning (RFT)** + **STaR** for math / reasoning
8. **DeepSeek-R1 distillation** — `R1 → Qwen-7B` recipe that beat GPT-4 on AIME
9. **Quality filters** — n-gram / embedding dedup · perplexity · diversity · `Nemotron-4 Reward`
10. The **2026 synthetic-data stack** + ethical / collapse / contamination risks


## 1 · Why synthetic data — the data wall

By 2024 frontier labs had **already trained on most of the open web**. Three forces pushed synthetic to the center:

| Force | What it means |
|---|---|
| **Data wall** | High-quality human text grows ~5%/yr; compute grows 10× faster. There's no more web to scrape. |
| **Cost of labels** | Human SFT pairs cost $1-10 each; LLM-generated pairs cost $0.001-0.01. **1000× cheaper.** |
| **Specialization** | Frontier-model output on *your* domain (code review, oncology notes, legal Q&A) is better than crowd-worker output. |
| **Capability transfer** | A strong teacher's reasoning trace bootstraps a small student to abilities the student couldn't have learned alone. This is the **Phi-3 / Orca / R1-distill** trick. |

> 📜 **Phi-3 thesis (Microsoft, 2024).** "Textbooks-quality synthetic data + careful filtering can take a **3.8B model** to GPT-3.5 territory on MMLU." Phi-3-mini scored 69 MMLU with ~3.3T tokens — half the tokens of Llama-3-8B. **Quality > quantity once you have a teacher.**

> 📜 **Orca-2 thesis (Microsoft, 2023).** "Don't just distill the *answer* — distill the *reasoning trace*. Tell the teacher *how* to solve, then train the student on (problem, reasoning, answer)." A 7B Orca-2 matched 70B Llama-2 on reasoning.


## 2 · The four flavors of distillation

```
                 ┌───────────────────────────────┐
                 │     SYNTHETIC DATA FLAVORS    │
                 └───────────────────────────────┘
                                 │
   ┌─────────────┬───────────────┴────────────────┬─────────────┐
   │             │                                │             │
SFT on        KL on              Self-Instruct          RLAIF / RFT
teacher       teacher                                  (use teacher
outputs       logits                                   as reward)
(Orca,        (DistilBERT,
 Phi)          MiniLM, R1-dist)
```

| Flavor | Supervision signal | When to use | Example |
|---|---|---|---|
| **SFT distillation** | `(prompt, teacher-completion)` pairs · CE loss | Universal default — works at any vocab / arch boundary | Phi-3 · Orca · R1→Qwen |
| **KL distillation** | KL(student ‖ teacher) on **logits** | Same tokenizer + arch family; gives much tighter fit | DistilBERT · MiniLM · TinyLlama |
| **Self-Instruct** | Teacher *generates the prompts too* (seed → expand) | Bootstrapping a domain from zero | Alpaca · WizardLM · Magpie |
| **RLAIF / RFT** | Teacher *judges* student outputs → reward / filter | Reasoning, alignment, math | DeepSeek-Math · STaR · Constitutional AI |

**Math — KL distillation:**

```
L = α · CE(student(x), y)              # hard-label CE
  + (1-α) · T² · KL( softmax(t/T) ‖ softmax(s/T) )
                                       # temperature-T softened logits
```

`T=2-4` is typical; the `T²` factor restores the gradient magnitude after softening.

**Math — SFT distillation** is just `CE(student, teacher_completion)` — no logit access required, which is why it works across API-only teachers (GPT-4, Claude, Gemini).


## 3 · Self-Instruct + Evol-Instruct

**Self-Instruct (Wang et al., 2022)** was the first scalable synthetic-data pipeline:

```
1. Hand-write 175 seed tasks
2. For each round:
   2a. Sample 6 seed tasks + 2 generated tasks
   2b. Prompt LLM:  "Here are 8 example tasks. Write 8 more."
   2c. ROUGE-L filter — drop new tasks > 0.7 similar to existing
   2d. LLM classifies "is this a classification task?" → branch
   2e. LLM generates input + output
3. Add survivors to the pool. Repeat for ~52k tasks.
```

This is exactly how **Stanford Alpaca** was built: 175 seeds → 52k instructions → Llama-7B SFT for $600.

**Evol-Instruct (WizardLM, 2023)** improved it with **prompt evolution** — five rewrite operators:

| Operator | What it does | Example |
|---|---|---|
| **Add constraints** | Inject extra requirements | "...in fewer than 50 words" |
| **Deepen** | Demand more detail | "...with worked examples and complexity analysis" |
| **Concretize** | Replace abstract → specific | "an animal" → "a Galápagos tortoise" |
| **Increase reasoning** | Convert one-step → multi-step | "What is X?" → "Compare X and Y, then conclude Z" |
| **Complicate input** | Add noisy / adversarial inputs | "Same task but the input is a JSON blob inside a markdown table" |

Run each operator with probability ⅕ → instruction tree grows in difficulty. **WizardLM-70B** was the first open chat model to beat GPT-3.5 on MT-Bench.


In [ ]:
# Self-Instruct seed → expansion (sketch)
SEED = [
    {"instruction": "Convert a CSV row to JSON.", "input": "name,age\nAda,36", "output": '{"name": "Ada", "age": 36}'},
    {"instruction": "Explain photosynthesis in one paragraph.", "input": "", "output": "..."},
    # ...173 more
]

PROMPT_TEMPLATE = '''You are a creative task author. Here are example tasks:

{examples}

Now write 8 MORE tasks of the same style. Be diverse — include classification, generation, reasoning, summarization, code.
Output as JSON list with keys "instruction", "input", "output".'''

def expand(pool, llm, rounds=100, k=6):
    import random, json
    for _ in range(rounds):
        examples = random.sample(pool, k)
        text = PROMPT_TEMPLATE.format(examples=json.dumps(examples, indent=2))
        new = llm(text)                       # generates 8 tasks
        for t in new:
            if max(rouge_l(t['instruction'], p['instruction']) for p in pool) < 0.7:
                pool.append(t)
    return pool


## 4 · Magpie — the zero-prompt synthetic-data trick

**Magpie (Xu et al., 2024)** showed something wild: you can **extract** the entire training distribution of an instruction-tuned LLM by feeding it *only the chat template prefix* and letting it free-generate the user turn:

```
<|user|>          ← give the model ONLY this
                  ← it now generates a plausible user instruction
↓
"How do I sort a list of dicts by a nested key in Python?"
                  ← then close the user tag and let it answer
<|assistant|>
"Use sorted(...) with a lambda..."
```

**Why it works:** instruction-tuned models have memorized the *manifold* of plausible user turns. Sampling from `<|user|>` is sampling from that manifold.

**Magpie pipeline (one Llama-3-70B-Instruct, no human prompts):**
1. 1M times: prompt with chat-template prefix → harvest a user instruction
2. 1M times: prompt with that instruction → harvest the assistant reply
3. Filter (quality model, reward model, length, language ID)
4. Result: **300k high-quality (instruction, response) pairs** for **$0 in human labels**

**Magpie-Air-3M (the public dump)** outperforms Alpaca, WizardLM, and OpenAssistant on Arena-Hard — purely synthetic, zero seeds.


## 5 · Orca / Orca-2 — distill the reasoning, not just the answer

The Orca recipe is one line: **"Tell GPT-4 to *think step by step and show its work*, then train the student on the trace."**

| Standard SFT pair | Orca SFT pair |
|---|---|
| `Q: 23×47?  A: 1081` | `Q: 23×47?  A: Let me compute: 23×47 = 23×40 + 23×7 = 920 + 161 = 1081.` |

Two tricks:
1. **System-message diversity** — 16 different "personas" (concise · detailed · pedagogical · scientist · adversarial) so the student sees many reasoning styles
2. **Cautious reasoning** — the system message instructs the teacher to "be wrong sometimes, then self-correct" — teaches the student to *backtrack*

**Orca-2-7B beat Llama-2-70B on AGIEval, BBH, MMLU-reasoning.** This is the same trick **DeepSeek-R1-Distill-Qwen-7B** used in 2025 — distill the `<think>` traces, not just the final answer.


## 6 · Self-Rewarding LM + RLAIF as data engines

**Self-Rewarding LM (Meta, 2024)** noticed: if you have one model, you can use it as **both** generator *and* judge:

```
Loop:
  step 1:  generate K candidate completions for each prompt
  step 2:  same model rates them with an "LLM-as-judge" prompt → reward
  step 3:  DPO on (best, worst) pairs from same prompt
  step 4:  the now-better model becomes the next round's judge
```

After 3 iterations a Llama-2-70B variant matched Claude 2 / GPT-4 on AlpacaEval — no extra human data, no external reward model. **The model is its own data engine.**

**RLAIF (Bai et al., Anthropic 2022 → 2024)** — same idea but the judge is *another model* (often the teacher) and the reward is used for **PPO / DPO** rather than just filtering. Constitutional AI (covered in M89) is a structured RLAIF recipe with a *written constitution* as the judging rubric.

> ⚠️ **Mode collapse risk.** Self-Reward + DPO can over-optimize for "tone the judge likes" and collapse diversity. Always evaluate on **held-out** benchmarks, not the model's own judge.


## 7 · Rejection Sampling Fine-Tuning (RFT) + STaR

For **verifiable** tasks (math, code, formal proofs) you don't need an LLM judge — you have a **ground-truth checker**. That makes synthetic data trivially correct.

**RFT pipeline (Touvron et al. 2023 · Llama-2 paper):**
```
1. Take a prompt with a known answer (GSM8K, MATH, HumanEval)
2. Sample N=100 chain-of-thought completions at T=0.8
3. Keep only completions whose final answer matches ground truth
4. SFT the student on the survivors
```

**STaR (Zelikman et al., 2022)** generalizes this with **rationalization** — when the model is wrong, prompt it with the *correct answer* and ask it to write a rationale that arrives there, then train on that rationale.

**Quiet-STaR (2024)** runs this *during pretraining* — generate hidden `<think>` tokens between every visible token, reward those that lower next-token loss. Llama-3 + Quiet-STaR gained 5 points on GSM8K with no extra data.

> 🧮 **DeepSeek-Math-7B** used RFT on ~700k GSM8K-like problems generated by GPT-4, filtered against a calculator + SymPy verifier. It hit 64% on MATH — beating GPT-3.5 and matching Llama-2-70B at 1/10 the size.


## 8 · DeepSeek-R1 distillation — the 2025 shocker

DeepSeek-R1 (Jan 2025) is a 671B MoE that learned long-form reasoning via **pure RL** (GRPO) with verifiable rewards — no SFT cold start. The shocker for the open-source world was the **distillation appendix**:

```
DeepSeek-R1 (671B teacher)
  │
  ├─→ generate 800k (prompt, reasoning, answer) traces
  │
  └─→ SFT Qwen-7B / Llama-8B / Qwen-32B on those traces
            │
            └─→ "R1-Distill-Qwen-7B" hits 55 on AIME 2024
                — beating GPT-4o (9.3) and Claude-3.5-Sonnet (16)
                — at 1/100 the parameters
```

The recipe is **boring SFT** — no RL, no logits, no special loss. Just `(prompt, <think>...</think><answer>...</answer>)` pairs and a vanilla cross-entropy loss. **The teacher's reasoning trace is the supervision signal.** This is the most-replicated paper of 2025; every lab now has an `R1-style-distill` SKU.

**Key practical takeaway:** if you have API access to a strong reasoner (R1, o1, o3, Claude-3.7-Thinking, Gemini-2-Thinking), you can build a domain-specialist 7B model for thousands of dollars, not millions.


In [ ]:
# DeepSeek-R1-style distillation recipe (HuggingFace TRL · sketch)
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Generate traces from the teacher (offline, batched, vLLM)
#    Each row: { 'prompt': "...", 'completion': "<think>...</think><answer>...</answer>" }
ds = Dataset.from_json('r1_traces_800k.jsonl')

# 2. Train the student
student = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-7B')
tok     = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-7B')

cfg = SFTConfig(
    output_dir='qwen7b-r1-distill',
    max_seq_length=16384,        # long reasoning traces!
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,
    learning_rate=1e-5,
    num_train_epochs=2,
    bf16=True,
    packing=True,                # critical for throughput
)
trainer = SFTTrainer(student, args=cfg, train_dataset=ds, tokenizer=tok)
trainer.train()


## 9 · Quality filters — the bit that actually matters

Naive synthetic data is **worse** than no data — repetition, hallucinated facts, format collapse all destroy a student. Every successful pipeline above leans on aggressive filtering:

| Filter | What it catches | Common tool |
|---|---|---|
| **Exact-dedup** | Verbatim copies (n-gram hash) | `text-dedup` · MinHash-LSH |
| **Semantic-dedup** | Near-duplicates | embedding sim > 0.9 (BGE / E5 / Nomic) |
| **Length** | One-line answers, runaways | percentile cuts on tokens |
| **Perplexity** | Garbled / non-fluent | KenLM or a small reference LM |
| **Language ID** | Off-target language slip | fastText / CLD3 |
| **Reward model** | Low-quality answers | **Nemotron-4-340B-Reward** · **Skywork-Reward** · ArmoRM |
| **Verifier** | Wrong answer (math/code) | calculator · SymPy · pytest · interpreter |
| **Diversity** | Mode collapse | embedding k-means → reweight rare clusters |
| **Toxicity / PII** | Unsafe outputs | Llama-Guard-3 · Detoxify |
| **Contamination** | Bench leak | n-gram overlap vs MMLU / GSM8K / HumanEval |

> 📏 **Rule of thumb (FineWeb-Edu, Phi-3, Nemotron reports):** filtering keeps **5-30%** of raw synthetic data. The 70-95% you throw away is what makes the rest work.


## 10 · The 2026 synthetic-data stack + risks

**OSS synthetic-data toolkit (May 2026):**

| Tool | What it does |
|---|---|
| **distilabel** (Argilla) | End-to-end synthetic-data pipelines; ships Self-Instruct, Evol-Instruct, Magpie, UltraFeedback templates |
| **Nemotron-4 340B + Reward** (NVIDIA) | Open weight teacher + reward model designed for synthetic-data work |
| **Magpie / Magpie-Air** (UW) | Public 1-3M Llama-3-70B dump; drop-in SFT corpus |
| **UltraFeedback / UltraChat** (THU) | 64k GPT-4 preference pairs / 1.5M GPT-3.5 chats |
| **OpenHermes-2.5 / SlimOrca / Capybara** | Curated mixtures used in nearly every open chat fine-tune |
| **Skywork-Reward / ArmoRM** | Best open reward models for RLHF/DPO filtering |
| **vLLM / SGLang / TensorRT-LLM** | High-throughput batch inference for teacher generation (M71 callback) |
| **Argilla / Lilac / DataTrove** | Human-in-the-loop curation + filtering at scale |

**The risks:**

1. **Model collapse (Shumailov et al., 2024).** Training repeatedly on a model's own outputs causes the tails of the distribution to vanish. Mitigation: always mix synthetic with fresh human data.
2. **Contamination.** Teacher may have memorized the benchmark; student inherits the leak. Always n-gram check against MMLU / GSM8K / HumanEval.
3. **Bias amplification.** Teacher's biases become the student's biases at higher concentration.
4. **License / TOS.** OpenAI / Anthropic / Google ToS prohibit "using outputs to develop a competing model." Use OSS teachers (Llama, Qwen, DeepSeek, Nemotron) when distilling for commercial release.
5. **Evaluation gaming.** Self-judged metrics drift up; held-out human eval drifts flat. Always hold something out.

> 🔮 **Where this is going.** Reasoning models (o1, R1, Claude-Thinking, Gemini-2-Thinking) generate **multi-thousand-token rationales** as a free byproduct. By 2026 the median open SFT corpus is **80% synthetic, 20% human-curated** — the inverse of 2022. M89 next on how to keep this aligned.


## ✅ Recap

- **Synthetic data** broke the data wall; **quality > quantity** once you have a strong teacher.
- Four flavors: **SFT distillation · KL distillation · Self-Instruct · RLAIF/RFT** — pick by what you have access to.
- **Self-Instruct → Evol-Instruct → Magpie** is the open-prompt evolution chain; **Orca / R1-distill** is the closed-prompt reasoning-trace chain.
- **RFT / STaR** for verifiable tasks (math, code, proofs) — the verifier *is* the reward model.
- **DeepSeek-R1 distillation**: boring SFT on `<think>...</think>` traces beats clever losses; replicated everywhere in 2025.
- **Filtering keeps 5-30%** of raw synthetic data — dedup · perplexity · reward model · verifier · diversity · contamination check.
- 2026 stack: **distilabel · Magpie · Nemotron-4 · vLLM · Skywork-Reward · Argilla**.
- Risks: **collapse · contamination · bias · ToS · self-eval drift** — always mix human data, always hold out human eval.

Next: **M89 — Constitutional AI / RLAIF Deep Dive**.
